# Perimotor PSTHs
Browse single-neuron activity aligned to **movement onset**, separately for
tracking (red) and playback (black). Display: **[-0.5, +0.5) s**, **5 ms** bins, Gaussian smoothing set by `SMOOTHING_SIGMA_S` below.
For each bout and neuron, subtract its own mean firing rate in **[-1, -0.7) s**
from its entire trace. Baseline is computed from raw counts before smoothing.
Then smooth each corrected bout and calculate the condition mean and **SEM across bouts**.
Negative values mean activity below that bout's baseline. No significance test is applied.
Set `SMOOTHING_SIGMA_S = 0` to disable smoothing. This is a configurable display
choice, not a replication of a verified paper method. A symmetric Gaussian can
spread post-onset changes into nearby pre-onset times (kernel support extends to 4 sigma).
Rasters always show the original, uncorrected, unsmoothed counts.

The saved premotor extraction ends at onset. This notebook instead reuses saved
bout timings and reads original spike counts, keeping only one source collection
in memory. It does not write or replace any derived data.

Run top to bottom using the project's `.venv` kernel. Paths work from the workspace,
repository, or notebook directory. Session and neuron controls redraw static figures.

In [ ]:
from pathlib import Path
import json
import pickle
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

cwd = Path.cwd().resolve()
repo_dir = next(
    (candidate for parent in (cwd, *cwd.parents)
     for candidate in (parent, parent / "etheremin_project")
     if (candidate / "movement_bouts" / "psth.py").is_file()),
    None,
)
if repo_dir is None:
    raise FileNotFoundError("Run from the eTheremin workspace or a directory inside it.")
sys.path.insert(0, str(repo_dir / "movement_bouts"))
import importlib
import psth
importlib.reload(psth)
from psth import extract_neuron_counts, plot_neuron_psth

data_dir = repo_dir.parent / "data"
bout_dir = data_dir / "derived" / "movement_bouts"
movement_bouts = pd.read_pickle(bout_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(bout_dir / "session_info.pkl")
parameters = json.loads((bout_dir / "parameters.json").read_text(encoding="utf-8"))
DT = parameters["DT"]
WINDOW_S = (-1.2, 1.0)  # Full extraction includes baseline and smoothing margins.
DISPLAY_WINDOW_S = (-0.5, 0.5)
BASELINE_WINDOW_S = (-1.0, -0.7)  # Per-bout mean, computed from raw counts.
BIN_WIDTH_S = 0.005    # Native 5 ms bins; must divide the extraction duration.
SMOOTHING_SIGMA_S = 0.01  # Gaussian sigma in seconds; 0 disables smoothing.
print(f"Loaded {len(movement_bouts):,} bouts across {len(session_info)} sessions; dt={DT:g} s.")

## Recording access
The cache holds the original neural recording collection and its neuron IDs.
Switching neurons or sessions in the same collection requires no reload. Switching
animal/headstage collections releases the previous collection before loading the next.
Only the selected neuron's per-bout windows are copied for plotting.

In [ ]:
recording_cache = {}

def load_recording(idx):
    info = session_info.loc[idx]
    source = info["feature_file"].replace("_feature_ss", "_data_ss")
    if recording_cache.get("source") != source:
        recording_cache.clear()
        print(f"Loading {source} ...")
        with (data_dir / source).open("rb") as stream:
            recordings = pickle.load(stream)
        id_file = info["feature_file"].replace("_feature_ss", "_neuron_idx_ss")
        with (data_dir / id_file).open("rb") as stream:
            ids = pickle.load(stream)
        recording_cache.update(source=source, recordings=recordings, ids=ids)
    session = int(info["session"])
    neural = recording_cache["recordings"][session]
    ids = recording_cache["ids"][session]
    if neural.shape != (len(ids), int(info["n_bins"])):
        raise ValueError(f"Recording dimensions do not match session_info for idx={idx}.")
    return neural, ids

## Browse neurons
Choose a session, then a neuron. Sliders update on release. Direction filtering
uses the movement's initial direction (-1 left, +1 right), not starting position.
Rasters show occupied original 5 ms bins; marker size reflects count when a bin
contains multiple spikes. Bout order is chronological within each condition.

Condition is read at onset, matching the saved bout table. Windows crossing a
condition/block transition are not separately filtered. Windows extending beyond
recording bounds are excluded and counted. The first 1.2 s precedes movement after
qualifying immobility; activity after onset can include movement, sound changes,
and subsequent behavior. These displays are a descriptive sanity check for
modulation around onset, not evidence of motor specificity or significance.

In [ ]:
session_selector = widgets.Dropdown(
    options=[(f"idx={idx}: {row.animal}, hs {row.headstage}, session {row.session}", idx)
             for idx, row in session_info.iterrows()],
    description="Session:", layout=widgets.Layout(width="650px"),
)
neuron_slider = widgets.IntSlider(value=0, min=0, max=0, description="Neuron:",
                                  continuous_update=False)
direction_selector = widgets.Dropdown(
    options=[("All directions", None), ("Left (-1)", -1), ("Right (+1)", 1)],
    description="Direction:",
)
plot_output = widgets.Output()
browser_state = {"updating": False}

def redraw_psth(change=None):
    if browser_state["updating"]:
        return
    with plot_output:
        clear_output(wait=True)
        try:
            idx = session_selector.value
            neural, ids = load_recording(idx)
            browser_state["updating"] = True
            try:
                neuron_slider.max = len(ids) - 1
                neuron_slider.value = min(neuron_slider.value, len(ids) - 1)
            finally:
                browser_state["updating"] = False
            neuron = neuron_slider.value
            bouts = movement_bouts.loc[
                movement_bouts["idx"].eq(idx) & movement_bouts["condition"].isin([0, 1])
            ].sort_values("onset_bin")
            if direction_selector.value is not None:
                bouts = bouts.loc[bouts["initial_direction"].eq(direction_selector.value)]
            counts, labels, excluded = extract_neuron_counts(
                neural, bouts, neuron=neuron, dt=DT, window_s=WINDOW_S,
            )
            print(f"{len(labels)} complete bouts; {len(excluded)} excluded at recording boundaries.")
            fig, axes = plot_neuron_psth(
                counts, labels, dt=DT, window_s=WINDOW_S, bin_width_s=BIN_WIDTH_S,
                title=f"idx={idx} | neuron {neuron}: {ids[neuron]}",
                display_window_s=DISPLAY_WINDOW_S, baseline_window_s=BASELINE_WINDOW_S,
                smoothing_sigma_s=SMOOTHING_SIGMA_S,
            )
            display(fig)
            plt.close(fig)
        except Exception as error:
            print(f"{type(error).__name__}: {error}")
            raise

for control in (session_selector, neuron_slider, direction_selector):
    control.observe(redraw_psth, names="value")
display(widgets.VBox([session_selector, widgets.HBox([neuron_slider, direction_selector]), plot_output]))
redraw_psth()

## Average PSTH across all neurons
First average the baseline-corrected, smoothed bout traces **within each neuron
and condition**, then take an **equally weighted mean of those neuron PSTHs**.
This uses the same windows and smoothing settings as the browser. There is no
z-scoring or division by firing rate. Shading is **SEM across neurons**, rather
than the single-neuron browser's SEM across bouts.

By default, include all saved sessions and all movement directions. The settings
below are independent of the browser selection. A neuron is one recorded unit in
one session; this does not establish distinct biological neurons across sessions.
Neurons without complete bouts for a condition are omitted from that condition.
The SEM describes variation among recorded unit PSTHs and does not account for
clustering within sessions or animals.

Run the computation cell once; it loads collections in turn, retains only the
per-neuron mean traces, and prints progress. Rerun it after changing any processing
settings. The plotting cell reuses the computed traces without loading recordings.

In [ ]:
from psth import session_neuron_psths, plot_population_psth

POPULATION_SESSION_INDICES = session_info.index.tolist()  # e.g. [0] for one session.
POPULATION_DIRECTION = None  # None = all; -1 = left; +1 = right.
if POPULATION_DIRECTION not in (None, -1, 1):
    raise ValueError("Choose None, -1 or +1 for POPULATION_DIRECTION.")
if not POPULATION_SESSION_INDICES or len(set(POPULATION_SESSION_INDICES)) != len(POPULATION_SESSION_INDICES):
    raise ValueError("Select at least one session, without repeated session indices.")

# Snapshot settings so the plot accurately describes the computed traces.
population_settings = dict(
    dt=DT, window_s=WINDOW_S, bin_width_s=BIN_WIDTH_S,
    baseline_window_s=BASELINE_WINDOW_S, smoothing_sigma_s=SMOOTHING_SIGMA_S,
)
population_traces = {0: [], 1: []}
population_units = {0: [], 1: []}
population_session_rows = []
for session_idx in POPULATION_SESSION_INDICES:
    neural, ids = load_recording(session_idx)
    bouts = movement_bouts.loc[
        movement_bouts["idx"].eq(session_idx) & movement_bouts["condition"].isin([0, 1])
    ].sort_values("onset_bin")
    if POPULATION_DIRECTION is not None:
        bouts = bouts.loc[bouts["initial_direction"].eq(POPULATION_DIRECTION)]
    traces, population_centers, bout_counts = session_neuron_psths(
        neural, bouts, **population_settings,
    )
    for condition in (0, 1):
        population_traces[condition].append(traces[condition])
        if len(traces[condition]):
            population_units[condition].extend((session_idx, neuron_id) for neuron_id in ids)
        population_session_rows.append(dict(
            idx=session_idx, condition=condition, n_neurons=len(traces[condition]),
            n_bouts=bout_counts[condition], n_excluded=len(bouts) - sum(bout_counts.values()),
        ))
    print(f"idx={session_idx}: {len(ids)} neurons; complete bouts tracking={bout_counts[0]}, playback={bout_counts[1]}")

# Each row is a single neuron's condition-averaged PSTH, not a single bout.
population_neuron_psths = {
    condition: pd.DataFrame(
        np.concatenate(population_traces[condition], axis=0),
        index=pd.MultiIndex.from_tuples(population_units[condition], names=["idx", "neuron_id"]),
        columns=pd.Index(population_centers, name="time_s"),
    )
    for condition in (0, 1)
}
population_session_summary = pd.DataFrame(population_session_rows)
population_direction = POPULATION_DIRECTION
print({condition: len(table) for condition, table in population_neuron_psths.items()}, "neurons per condition")

In [ ]:
settings = population_settings
baseline = settings["baseline_window_s"]
baseline_text = f"baseline [{baseline[0]:g}, {baseline[1]:g}) s" if baseline is not None else "raw rates"
direction_text = {None: "all directions", -1: "left", 1: "right"}[population_direction]
n_sessions = population_session_summary["idx"].nunique()
fig_population, ax_population = plot_population_psth(
    population_neuron_psths, population_centers,
    display_window_s=DISPLAY_WINDOW_S,
    baseline_subtracted=baseline is not None,
    smoothed=settings["smoothing_sigma_s"] > 0,
    title=(f"Population PSTH | {n_sessions} sessions | {direction_text}\n"
           f"{baseline_text} | {settings['bin_width_s'] * 1000:g} ms bins | "
           f"Gaussian sigma={settings['smoothing_sigma_s'] * 1000:g} ms"),
)
plt.show()

## Population PSTHs split by neuron type
Load `data/df_clus_encoding_phd.pkl` and match its **neuron_idx** to the full
**neuron_id** in each population PSTH row, using the supplied mapping:
Motor = 0, Sensory = 1, Sensorimotor = 2, None = 3.

Each panel averages unit PSTHs equally within its type and condition, with
**SEM across units**, the same baseline/smoothing, and shared axis scales.
These cells reuse `population_neuron_psths`; no original recordings are loaded.
They respect the population section's session and direction selection.

**None is an assigned category.** Units absent from the classification table
remain unclassified and are excluded from these four panels; they remain in the
all-neuron average above. Coverage and excluded IDs are available below. Changing
neuron types does not change or refit any PSTH. Rerun the population calculation
first if changing preprocessing, sessions, or movement-direction selection.

In [ ]:
df_clus_encoding_phd = pd.read_pickle(data_dir / "df_clus_encoding_phd.pkl")
category_to_cluster = {"Motor": 0, "Sensory": 1, "Sensorimotor": 2, "None": 3}
cluster_to_category = {cluster: category for category, cluster in category_to_cluster.items()}
if df_clus_encoding_phd["neuron_idx"].isna().any() or df_clus_encoding_phd["neuron_idx"].duplicated().any():
    raise ValueError("Classification neuron_idx must be nonmissing and unique.")
if not df_clus_encoding_phd["cluster_encoding"].isin(cluster_to_category).all():
    raise ValueError("Classification contains missing or unknown cluster_encoding values.")
cluster_by_neuron = df_clus_encoding_phd.set_index("neuron_idx")["cluster_encoding"]
neuron_types_by_condition = {}
coverage_rows = []
unclassified_units = {}
for condition, table in population_neuron_psths.items():
    # Match exact IDs, preserving the PSTH index and its row order.
    categories = table.index.get_level_values("neuron_id").map(cluster_by_neuron).map(cluster_to_category)
    labels = pd.Series(categories, index=table.index, name="neuron_type")
    neuron_types_by_condition[condition] = labels
    unclassified_units[condition] = table.index[labels.isna()]
    coverage_rows.append({
        "condition": {0: "Tracking", 1: "Playback"}[condition],
        "total_units": len(table),
        **{category: int(labels.eq(category).sum()) for category in category_to_cluster},
        "unclassified_excluded": int(labels.isna().sum()),
    })
neuron_type_coverage = pd.DataFrame(coverage_rows).set_index("condition")
display(neuron_type_coverage)

In [ ]:
# Reload permits running just this new section in an existing notebook kernel.
importlib.reload(psth)
from psth import plot_population_psth_by_type

settings = population_settings
baseline = settings["baseline_window_s"]
baseline_text = f"baseline [{baseline[0]:g}, {baseline[1]:g}) s" if baseline is not None else "raw rates"
direction_text = {None: "all directions", -1: "left", 1: "right"}[population_direction]
n_sessions = population_session_summary["idx"].nunique()
fig_by_type, axes_by_type = plot_population_psth_by_type(
    population_neuron_psths, neuron_types_by_condition, population_centers,
    display_window_s=DISPLAY_WINDOW_S,
    baseline_subtracted=baseline is not None,
    smoothed=settings["smoothing_sigma_s"] > 0,
    title=(f"Population PSTHs by neuron type | {n_sessions} sessions | {direction_text}\n"
           f"{baseline_text} | {settings['bin_width_s'] * 1000:g} ms bins | "
           f"Gaussian sigma={settings['smoothing_sigma_s'] * 1000:g} ms"),
)
plt.show()